
# Mental Health Late Fusion Model

Notebook ini menggunakan:
- Deep Learning untuk data tabular
- NLP untuk data teks
- Late Fusion Adjustment
- TensorFlow Functional API
- Custom Layer

Tanpa Transformer / IndoBERT.


In [137]:
# =========================================================
# 1. IMPORT LIBRARY
# =========================================================

import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Input,
    Embedding,
    LSTM,
    Bidirectional,
)

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.15.0


In [ ]:
# =========================================================
# LOAD DATASET ASLI
# =========================================================

tabular_df = pd.read_csv("../data/survey.csv")
text_df = pd.read_csv("../data/mental_heath_unbanlanced.csv")

print("Tabular Shape :", tabular_df.shape)
print("Text Shape    :", text_df.shape)

# =========================================================
# LOAD FULL DATASET
# =========================================================

# gunakan seluruh data asli tanpa sampling
print("\n===== AFTER LOADING DATA =====")

print("Tabular :", tabular_df.shape)
print("Text    :", text_df.shape)

print("\nTabular Preview")
print(tabular_df.head())

print("\nText Preview")
print(text_df.head())

import re

def clean_text(text):

    text = str(text)

    # hapus emoji
    text = re.sub(
        r'[^\x00-\x7F]+',
        ' ',
        text
    )

    # hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# apply ke dataset text
text_df["text"] = text_df["text"].apply(clean_text)

print("Cleaning selesai!")

Tabular Shape : (1259, 27)
Text Shape    : (49612, 3)

===== AFTER LOADING DATA =====
Tabular : (1259, 27)
Text    : (49612, 3)

Tabular Preview
             Timestamp  Age  Gender         Country state self_employed  \
0  2014-08-27 11:29:31   37  Female   United States    IL           NaN   
1  2014-08-27 11:29:37   44       M   United States    IN           NaN   
2  2014-08-27 11:29:44   32    Male          Canada   NaN           NaN   
3  2014-08-27 11:29:46   31    Male  United Kingdom   NaN           NaN   
4  2014-08-27 11:30:22   31    Male   United States    TX           NaN   

  family_history treatment work_interfere    no_employees  ...  \
0             No       Yes          Often            6-25  ...   
1             No        No         Rarely  More than 1000  ...   
2             No        No         Rarely            6-25  ...   
3            Yes       Yes          Often          26-100  ...   
4             No        No          Never         100-500  ...   

       

In [139]:

# =========================================================
# 4. PREPROCESS TABULAR DATA
# =========================================================

target_tab = "treatment"

X_tab = tabular_df.drop(columns=[target_tab])
y_tab = tabular_df[target_tab]

encoders = {}

for col in X_tab.columns:

    le = LabelEncoder()

    X_tab[col] = le.fit_transform(
        X_tab[col].astype(str)
    )

    encoders[col] = le

label_tab = LabelEncoder()

y_tab = label_tab.fit_transform(y_tab)

y_tab_onehot = to_categorical(y_tab)

scaler = StandardScaler()

X_tab = scaler.fit_transform(X_tab)

X_tab_train, X_tab_test, y_tab_train, y_tab_test = train_test_split(
    X_tab,
    y_tab_onehot,
    test_size=0.2,
    random_state=42
)

print(X_tab_train.shape)


(1007, 26)


In [140]:

# =========================================================
# 5. PREPROCESS TEXT DATA
# =========================================================

TEXT_COLUMN = "text"
LABEL_COLUMN = "status"

X_text = text_df[TEXT_COLUMN].astype(str)

label_txt = LabelEncoder()

y_text = label_txt.fit_transform(
    text_df[LABEL_COLUMN]
)

y_txt_onehot = to_categorical(y_text)

X_txt_train, X_txt_test, y_txt_train, y_txt_test = train_test_split(
    X_text,
    y_txt_onehot,
    test_size=0.2,
    random_state=42
)

print(text_df.columns)
print(label_txt.classes_)


Index(['Unique_ID', 'text', 'status'], dtype='object')
['Anxiety' 'Depression' 'Normal' 'Suicidal']


In [141]:

# =========================================================
# 6. CUSTOM LAYER
# =========================================================

class AttentionLayer(layers.Layer):

    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):

        self.W = self.add_weight(
            shape=(input_shape[-1], 1),
            initializer="random_normal",
            trainable=True
        )

    def call(self, inputs):

        score = tf.nn.tanh(
            tf.matmul(inputs, self.W)
        )

        weights = tf.nn.softmax(score, axis=1)

        context = weights * inputs

        context = tf.reduce_sum(
            context,
            axis=1
        )

        return context


In [142]:
# =========================================================
# 7. TABULAR MODEL
# =========================================================

input_tab = layers.Input(shape=(X_tab_train.shape[1],), name="input_tab")

x1 = layers.Dense(128, activation="relu")(input_tab)
x1 = layers.Dropout(0.4)(x1)

x1 = layers.Dense(64, activation="relu")(x1)
x1 = layers.Dropout(0.3)(x1)

x1 = layers.Dense(32, activation="relu")(x1)

out_tab = layers.Dense(2, activation="softmax", name="out_tab")(x1)

model_tab = models.Model(input_tab, out_tab)

model_tab.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

model_tab.summary()

Model: "model_11"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_tab (InputLayer)      [(None, 26)]              0         
                                                                 
 dense_18 (Dense)            (None, 128)               3456      
                                                                 
 dropout_14 (Dropout)        (None, 128)               0         
                                                                 
 dense_19 (Dense)            (None, 64)                8256      
                                                                 
 dropout_15 (Dropout)        (None, 64)                0         
                                                                 
 dense_20 (Dense)            (None, 32)                2080      
                                                                 
 out_tab (Dense)             (None, 2)                 66 

In [143]:
# =========================================================
# EARLY STOPPING & LEARNING RATE SCHEDULER
# =========================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

In [144]:
# augment hanya TRAIN SET

def augment_tabular(X, y, noise_factor=0.001, n_aug=1):
    X = np.array(X)
    y = np.array(y)

    X_aug = [X]
    y_aug = [y]

    for _ in range(n_aug):
        noise = np.random.normal(loc=0.0, scale=noise_factor, size=X.shape)
        X_aug.append(X + noise)
        y_aug.append(y)

    X_aug = np.vstack(X_aug)
    y_aug = np.vstack(y_aug)
    return X_aug, y_aug

X_train_tab_aug, y_train_tab_aug = augment_tabular(
    X_tab_train,
    y_tab_train,
    noise_factor=0.001,
    n_aug=2
)

print("X_train asli :", X_tab_train.shape)
print("X_train aug  :", X_train_tab_aug.shape)


X_train asli : (1007, 26)
X_train aug  : (3021, 26)


In [145]:
# =========================================================
# 8. TRAIN TABULAR MODEL
# =========================================================

history_tab = model_tab.fit(
    X_train_tab_aug,
    y_train_tab_aug,
    validation_data=(X_tab_test, y_tab_test),
    epochs=40,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=2,
)

Epoch 1/40
95/95 - 1s - loss: 0.6530 - accuracy: 0.6216 - val_loss: 0.5955 - val_accuracy: 0.7381 - lr: 0.0010 - 800ms/epoch - 8ms/step
Epoch 2/40
95/95 - 0s - loss: 0.5845 - accuracy: 0.6981 - val_loss: 0.5707 - val_accuracy: 0.7381 - lr: 0.0010 - 154ms/epoch - 2ms/step
Epoch 3/40
95/95 - 0s - loss: 0.5559 - accuracy: 0.7216 - val_loss: 0.5482 - val_accuracy: 0.7500 - lr: 0.0010 - 178ms/epoch - 2ms/step
Epoch 4/40
95/95 - 0s - loss: 0.5310 - accuracy: 0.7362 - val_loss: 0.5291 - val_accuracy: 0.7421 - lr: 0.0010 - 239ms/epoch - 3ms/step
Epoch 5/40
95/95 - 0s - loss: 0.5094 - accuracy: 0.7544 - val_loss: 0.5123 - val_accuracy: 0.7460 - lr: 0.0010 - 242ms/epoch - 3ms/step
Epoch 6/40
95/95 - 0s - loss: 0.4839 - accuracy: 0.7590 - val_loss: 0.4903 - val_accuracy: 0.7579 - lr: 0.0010 - 227ms/epoch - 2ms/step
Epoch 7/40
95/95 - 0s - loss: 0.4469 - accuracy: 0.7918 - val_loss: 0.4816 - val_accuracy: 0.7579 - lr: 0.0010 - 231ms/epoch - 2ms/step
Epoch 8/40
95/95 - 0s - loss: 0.4346 - accuracy:

In [146]:
# =========================================================
# 9. NLP MODEL
# =========================================================

vectorize_layer = layers.TextVectorization(
    max_tokens=10000, output_mode="int", output_sequence_length=100
)

vectorize_layer.adapt(X_txt_train)

input_txt = layers.Input(shape=(1,), dtype=tf.string, name="input_txt")

x2 = vectorize_layer(input_txt)

x2 = layers.Embedding(10000, 128)(x2)

x2 = Bidirectional(LSTM(128, return_sequences=True))(x2)

x2 = AttentionLayer()(x2)

x2 = layers.Dense(64, activation="relu")(x2)

x2 = layers.Dropout(0.4)(x2)

out_txt = layers.Dense(4, activation="softmax", name="out_txt")(x2)

model_txt = models.Model(input_txt, out_txt)

model_txt.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

model_txt.summary()

Model: "model_12"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_txt (InputLayer)      [(None, 1)]               0         
                                                                 
 text_vectorization_5 (Text  (None, 100)               0         
 Vectorization)                                                  
                                                                 
 embedding_5 (Embedding)     (None, 100, 128)          1280000   
                                                                 
 bidirectional_5 (Bidirecti  (None, 100, 256)          263168    
 onal)                                                           
                                                                 
 attention_layer_10 (Attent  (None, 256)               256       
 ionLayer)                                                       
                                                          

In [147]:
# =========================================================
# 10. TRAIN NLP MODEL
# =========================================================

history_txt = model_txt.fit(
    np.array(X_txt_train).reshape(-1, 1),
    y_txt_train,
    validation_data=(np.array(X_txt_test).reshape(-1, 1), y_txt_test),
    epochs=25,
    batch_size=64,
    callbacks=[early_stopping, reduce_lr],
    verbose=2,
)

Epoch 1/25
621/621 - 185s - loss: 0.7349 - accuracy: 0.6824 - val_loss: 0.5601 - val_accuracy: 0.7664 - lr: 0.0010 - 185s/epoch - 299ms/step
Epoch 2/25
621/621 - 315s - loss: 0.5161 - accuracy: 0.7945 - val_loss: 0.5507 - val_accuracy: 0.7780 - lr: 0.0010 - 315s/epoch - 508ms/step
Epoch 3/25
621/621 - 238s - loss: 0.4478 - accuracy: 0.8254 - val_loss: 0.5498 - val_accuracy: 0.7773 - lr: 0.0010 - 238s/epoch - 384ms/step
Epoch 4/25
621/621 - 389s - loss: 0.3920 - accuracy: 0.8474 - val_loss: 0.5919 - val_accuracy: 0.7596 - lr: 0.0010 - 389s/epoch - 626ms/step
Epoch 5/25
621/621 - 256s - loss: 0.3454 - accuracy: 0.8678 - val_loss: 0.6574 - val_accuracy: 0.7586 - lr: 0.0010 - 256s/epoch - 412ms/step
Epoch 6/25
621/621 - 349s - loss: 0.2687 - accuracy: 0.8995 - val_loss: 0.6949 - val_accuracy: 0.7591 - lr: 5.0000e-04 - 349s/epoch - 563ms/step
Epoch 7/25
621/621 - 330s - loss: 0.2315 - accuracy: 0.9161 - val_loss: 0.7929 - val_accuracy: 0.7599 - lr: 5.0000e-04 - 330s/epoch - 531ms/step
Epoch

In [148]:
# =========================================================
# 11. EVALUATE TABULAR MODEL & NLP MODEL
# =========================================================

loss_tab, acc_tab = model_tab.evaluate(X_tab_test, y_tab_test)
print("Tabular Accuracy:", acc_tab)
print("Tabular Training Accuracy:", max(history_tab.history['accuracy']))

loss_txt, acc_txt = model_txt.evaluate(
    np.array(X_txt_test).reshape(-1,1),
    y_txt_test
)
print("Text Accuracy:", acc_txt)
print("Text Training Accuracy:", max(history_txt.history['accuracy']))

8/8 [==============================] - 0s 2ms/step - loss: 0.4245 - accuracy: 0.7937
Tabular Accuracy: 0.7936508059501648
Tabular Training Accuracy: 0.8699105978012085
311/311 [==============================] - 39s 127ms/step - loss: 0.5498 - accuracy: 0.7773
Text Accuracy: 0.777285099029541
Text Training Accuracy: 0.9349693655967712


In [149]:

# =========================================================
# 17. SAVE MODEL
# =========================================================

model_tab.save("tabular_model.keras")
model_txt.save("text_model.keras")

print("Model berhasil disimpan!")


Model berhasil disimpan!


In [150]:

# =========================================================
# 17. LOAD MODEL
# =========================================================

loaded_tab = tf.keras.models.load_model(
    "tabular_model.keras"
)

loaded_txt = tf.keras.models.load_model(
    "text_model.keras",
    custom_objects={
        "AttentionLayer": AttentionLayer
    }
)

print("Model berhasil di-load kembali!")


Model berhasil di-load kembali!


In [152]:
# =========================================================
# 18. LATE FUSION PREDICTION & ADJUSTMENT
# =========================================================

def run_late_fusion_prediction(user_text, tabular_dict=None, nlp_weight=0.7, tabular_weight=0.3):
    """
    Perform late fusion between the NLP mental health model and tabular treatment signal.
    The final result is designed to be broader than the 4-class NLP decision alone.

    Args:
        user_text (str): User input text describing condition.
        tabular_dict (dict, optional): Tabular feature values for treatment/screening.
        nlp_weight (float): Weight of NLP risk score.
        tabular_weight (float): Weight of tabular treatment signal.

    Returns:
        dict: Combined prediction, raw outputs, and adjusted recommendation.
    """

    text_clean = clean_text(user_text)

    # NLP inference
    prob_txt = loaded_txt.predict(np.array([text_clean]).reshape(-1, 1), verbose=0)[0]
    nlp_labels = list(label_txt.classes_)
    nlp_probs = dict(zip(nlp_labels, prob_txt))
    primary_nlp_label = nlp_labels[int(np.argmax(prob_txt))]

    # Determine a simple risk score from NLP: depression or suicidal risk
    if len(prob_txt) >= 4:
        nlp_risk = float(max(prob_txt[1], prob_txt[3]))
    else:
        nlp_risk = float(np.max(prob_txt))

    # Tabular inference
    tabular_probs = None
    treatment_confidence = None
    treatment_recommendation = "Unavailable"
    if tabular_dict is not None:
        cleaned_tabular = []
        for col_name in encoders.keys():
            value = tabular_dict.get(col_name, "")
            try:
                encoded = encoders[col_name].transform([str(value)])[0]
            except Exception:
                encoded = 0
            cleaned_tabular.append(encoded)

        arr = pd.DataFrame([cleaned_tabular], columns=list(encoders.keys()))
        arr = scaler.transform(arr)
        prob_tab = loaded_tab.predict(arr, verbose=0)[0]
        tabular_probs = dict(zip(label_tab.classes_.tolist(), prob_tab))
        treatment_confidence = float(prob_tab[1])
        treatment_recommendation = "Yes" if treatment_confidence >= 0.5 else "No"
    else:
        tabular_probs = {label: None for label in label_tab.classes_.tolist()}

    # Fuse text risk with tabular treatment signal
    if treatment_confidence is not None:
        fused_risk = nlp_weight * nlp_risk + tabular_weight * treatment_confidence
    else:
        fused_risk = nlp_risk

    fused_risk = float(np.clip(fused_risk, 0.0, 1.0))

    if fused_risk >= 0.65:
        final_state = "High risk - professional support recommended"
    elif fused_risk >= 0.35:
        final_state = "Moderate risk - monitor and consider support"
    else:
        final_state = "Low risk - likely stable"

    final_recommendation = "Treatment recommended" if fused_risk >= 0.5 else "No immediate treatment"

    return {
        "input_text": user_text,
        "cleaned_text": text_clean,
        "nlp_probs": nlp_probs,
        "primary_nlp_label": primary_nlp_label,
        "nlp_risk": nlp_risk,
        "tabular_probs": tabular_probs,
        "treatment_confidence": treatment_confidence,
        "treatment_recommendation": treatment_recommendation,
        "fused_risk": fused_risk,
        "final_state": final_state,
        "final_recommendation": final_recommendation,
    }


print("\n" + "="*70)
print("LATE FUSION PREDICTION & ADJUSTMENT READY")
print("="*70)
print("The new function is available as run_late_fusion_prediction().")
print("It combines NLP probability, treatment signal, and gives a broader decision.")


# =========================================================
# 19. LATE FUSION TESTING
# =========================================================

sample_tabular = tabular_df.drop(columns=[target_tab]).iloc[0].to_dict()
print("\nSample tabular features used for fusion:\n", sample_tabular)

examples = [
    (
        "Saya merasa sangat sedih, kehilangan semangat, dan mudah menangis.",
        None,
    ),
    (
        "Saya sering berpikir tidak ada gunanya hidup dan takut masa depan.",
        sample_tabular,
    ),
    (
        "Saya kadang cemas, tetapi masih bisa bekerja dan tidur normal.",
        None,
    ),
]

for idx, (text, tabular_input) in enumerate(examples, start=1):
    print("\n" + "*"*30)
    print(f"TEST {idx}")
    print("*"*30)
    result = run_late_fusion_prediction(text, tabular_input)
    print(f"Input text: {result['input_text']}")
    print(f"Cleaned text: {result['cleaned_text']}")
    print(f"Primary NLP label: {result['primary_nlp_label']}")
    print(f"NLP risk score: {result['nlp_risk']:.3f}")
    print(f"Fused risk score: {result['fused_risk']:.3f}")
    print(f"Final state: {result['final_state']}")
    print(f"Final recommendation: {result['final_recommendation']}")
    print(f"Treatment recommendation from tabular model: {result['treatment_recommendation']}")
    print("NLP probabilities:")
    for k, v in result['nlp_probs'].items():
        print(f"  {k}: {v*100:.1f}%")
    print("Tabular probabilities:")
    for k, v in result['tabular_probs'].items():
        print(f"  {k}: {v if v is None else f'{v*100:.1f}%'}")



LATE FUSION PREDICTION & ADJUSTMENT READY
The new function is available as run_late_fusion_prediction().
It combines NLP probability, treatment signal, and gives a broader decision.

Sample tabular features used for fusion:
 {'Timestamp': '2014-08-27 11:29:31', 'Age': 37, 'Gender': 'Female', 'Country': 'United States', 'state': 'IL', 'self_employed': nan, 'family_history': 'No', 'work_interfere': 'Often', 'no_employees': '6-25', 'remote_work': 'No', 'tech_company': 'Yes', 'benefits': 'Yes', 'care_options': 'Not sure', 'wellness_program': 'No', 'seek_help': 'Yes', 'anonymity': 'Yes', 'leave': 'Somewhat easy', 'mental_health_consequence': 'No', 'phys_health_consequence': 'No', 'coworkers': 'Some of them', 'supervisor': 'Yes', 'mental_health_interview': 'No', 'phys_health_interview': 'Maybe', 'mental_vs_physical': 'Yes', 'obs_consequence': 'No', 'comments': nan}

******************************
TEST 1
******************************
Input text: Saya merasa sangat sedih, kehilangan semangat